# Donna Model Comparison Tool

Compare conversation traces from two Donna experiment runs through human evaluation.

## Workflow

```
PART 1: CREATE COMPARISONS
────────────────────────────────────────────
  Step 1. Setup & Configuration
  Step 2. Connect to Dataloop
  Step 3. Load & Pair Conversation Logs
  Step 4. Create Comparison Items
          ├─ 4a. Define helper functions
          ├─ 4b. Download & transform traces
          └─ 4c. Upload to Evaluation Studio
  Step 5. Create Annotation Task

  ⏸️  PAUSE — Complete annotations on Dataloop platform

PART 2: ANALYZE RESULTS
────────────────────────────────────────────
  Step 6. Load Comparison Items
  Step 7. Collect Annotations
  Step 8. Summarize Results
  Step 9. Raw Data (Optional)
```

## Key Terms

| Term | Definition |
|------|------------|
| **Trace** | A conversation log from one Donna run (stored as JSON) |
| **Experiment Directory** | Folder containing traces from a specific model config |
| **Comparison Item** | Side-by-side view of two traces for the same query |

## Setup Note: Evaluation Studio Recipe

The **model-AB-testing** multimodal recipe was created manually in the Dataloop platform UI using the Evaluation Studio Form Builder.

**Important:** The recipe uses a custom JS file (`recipe/model-AB-testing.js`) to avoid validation errors from the default JS template. If recreating the recipe, paste this JS into the Form Builder's JS tab.

---
# PART 1: Create Comparisons
---
## Step 1: Setup & Configuration

Run the imports, then configure the experiment directories you want to compare.

In [1]:
import dtlpy as dl
import json
import io
import re
import pandas as pd
from datetime import datetime
from collections import defaultdict
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.std import tqdm

**Edit the values below** to specify which experiments to compare:

| Variable | Description |
|----------|-------------|
| `DIR_A` | Path to first experiment (Trace A) |
| `DIR_B` | Path to second experiment (Trace B) |
| `LAYOUT_NAME` | Evaluation Studio layout for rendering comparisons |

In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — Edit these values
# ═══════════════════════════════════════════════════════════════════════════════

# Dataloop project and dataset names
PROJECT_NAME = "Eddie"
SOURCE_DATASET_NAME = "conversation-logs"
OUTPUT_DATASET_NAME = "model-comparisons"

# Experiment directories to compare (paths within SOURCE_DATASET_NAME)
DIR_A = "/csvs/csv_20251223_101531"  # ← Trace A (e.g., old config)
DIR_B = "/csvs/csv_20251226_111400"  # ← Trace B (e.g., new config)

# # Evaluation Studio layout name
# RECIPE_name = "model-comparison"

# ═══════════════════════════════════════════════════════════════════════════════
# AUTO-GENERATED (no changes needed)
# ═══════════════════════════════════════════════════════════════════════════════

# TODO uncomment when notebook is live
date_str = datetime.now().strftime("%Y%m%d.%H%M") 
# date_str = ""
OUTPUT_DIR = f"/{date_str}_{Path(DIR_A).name}_vs_{Path(DIR_B).name}"

print("Configuration Summary")
print("=" * 50)
print(f"  Trace A: {DIR_A}")
print(f"  Trace B: {DIR_B}")
print(f"  Output:  {OUTPUT_DATASET_NAME}{OUTPUT_DIR}")
print("=" * 50)

Configuration Summary
  Trace A: /csvs/csv_20251223_101531
  Trace B: /csvs/csv_20251226_111400
  Output:  model-comparisons/20260104.1501_csv_20251223_101531_vs_csv_20251226_111400


---
## Step 2: Connect to Dataloop

Connects to your project and datasets. The output dataset will be created if it doesn't exist.

In [3]:
# Get project
project = dl.projects.get(project_name=PROJECT_NAME)
print(f"Project: {project.name}")

# Get source dataset (must exist)
source_dataset = project.datasets.get(dataset_name=SOURCE_DATASET_NAME)
print(f"Source dataset: {source_dataset.name}")

# Get or create output dataset
try:
    output_dataset = project.datasets.get(dataset_name=OUTPUT_DATASET_NAME)
    print(f"Found output dataset: {output_dataset.name}")
except:
    output_dataset = project.datasets.create(dataset_name=OUTPUT_DATASET_NAME)
    print(f"Created output dataset: {output_dataset.name}")

Project: Eddie
Source dataset: conversation-logs
Found output dataset: model-comparisons


---
## Step 3: Load & Pair Conversation Logs

Loads items from both directories and pairs them by query ID (e.g., `csv_query_001` in Dir A matches `csv_query_001` in Dir B).

In [4]:
# Load items from directory A (from source dataset)
filters_a = dl.Filters()
filters_a.add(field='dir', values=DIR_A)
filters_a.add(field='name', values='*csv_query_*')
items_a = list(source_dataset.items.list(filters=filters_a).all())
print(f"Directory A ({DIR_A}): {len(items_a)} items")

# Load items from directory B (from source dataset)
filters_b = dl.Filters()
filters_b.add(field='dir', values=DIR_B)
filters_b.add(field='name', values='*csv_query_*')
items_b = list(source_dataset.items.list(filters=filters_b).all())
print(f"Directory B ({DIR_B}): {len(items_b)} items")

Directory A (/csvs/csv_20251223_101531): 106 items
Directory B (/csvs/csv_20251226_111400): 106 items


Items are paired by extracting the query ID from filenames. Only queries that exist in **both** directories are included.

In [5]:
# Extract query ID from filename (e.g., "csv_query_001" from "csv_query_001.json")
def get_query_id(filename):
    match = re.match(r'(csv_query_\d+)', filename)
    return match.group(1) if match else None

# Build lookup dictionaries
dir_a_by_id = {get_query_id(item.name): item for item in items_a if get_query_id(item.name)}
dir_b_by_id = {get_query_id(item.name): item for item in items_b if get_query_id(item.name)}

# Find matching pairs
common_ids = sorted(set(dir_a_by_id.keys()) & set(dir_b_by_id.keys()))
pairs = [(dir_a_by_id[qid], dir_b_by_id[qid], qid) for qid in common_ids]

print(f"Found {len(pairs)} matching pairs")

# Report unmatched
unmatched_a = set(dir_a_by_id.keys()) - set(common_ids)
unmatched_b = set(dir_b_by_id.keys()) - set(common_ids)
if unmatched_a:
    print(f"Unmatched in A: {len(unmatched_a)}")
if unmatched_b:
    print(f"Unmatched in B: {len(unmatched_b)}")

Found 106 matching pairs


---
## Step 4: Create Comparison Items

Transforms paired traces into comparison items for Evaluation Studio. This step has three phases:

| Phase | Description |
|-------|-------------|
| **4a** | Define helper functions for data transformation |
| **4b** | Download traces and prepare comparison data |
| **4c** | Upload comparison items to Dataloop |

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 4a: HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════

MAX_WORKERS = 20  # Concurrent operations (adjust based on API limits)

def make_model_table(agents, label):
    """Generate markdown table summarizing agent models.
    
    Args:
        agents: Dict mapping agent name -> model name
        label: Display label (e.g., "Trace A") - not used in output, kept for compatibility
    
    Returns:
        Markdown string with agent configuration table
    """
    lines = ["| Agent | Model Used |", "|-------|------------|"]
    lines += [f"| **{name}** | {model} |" for name, model in agents.items()]
    if agents:
        lines.append(f"\n*Pipeline: {' → '.join(agents.keys())}*")
    return "\n".join(lines)


def extract_agent_models(trace_data):
    """Extract agent -> model mapping from trace data.
    
    Handles real trace format with models_by_agent:
        {"documentation": {"model": "openai/gpt-4o"}}
    
    Args:
        trace_data: Dict containing the trace JSON
    
    Returns:
        Dict mapping agent name -> model name
    """
    models_by_agent = trace_data.get("models_by_agent", {})
    return {
        agent: config.get("model", "unknown") 
        for agent, config in models_by_agent.items()
    }


def transform_tool_calls(tool_calls):
    """Transform tool calls into a readable format for comparison.
    
    Args:
        tool_calls: List of tool call dicts from trace
    
    Returns:
        List of simplified tool call dicts with key info
    """
    result = []
    for tc in tool_calls:
        result.append({
            "sequence": tc.get("sequence"),
            "tool_name": tc.get("tool_name"),
            "agent": tc.get("agent"),
            "input": tc.get("args"),
            "output": tc.get("output"),
            "status": tc.get("status"),
            "runtime_seconds": round(tc.get("runtime", 0), 2)
        })
    return result


def format_tool_calls_markdown(tool_calls: list, trace_label: str = "Trace") -> str:
    """Convert tool calls array to collapsible markdown format.

    Args:
        tool_calls: List of tool call dicts with keys like:
            - sequence, tool_name, agent, input/args, output, status, runtime_seconds
        trace_label: Label for the trace (e.g., "Trace A") - not used in output

    Returns:
        Markdown string with collapsible tool call details using <details>/<summary> tags
    """
    if not tool_calls:
        return "*No tool calls recorded*"

    lines = []

    for i, tc in enumerate(tool_calls, 1):
        tool_name = tc.get("tool_name", "unknown")
        agent = tc.get("agent", "unknown")
        args = tc.get("input") or tc.get("args", "{}")
        output = tc.get("output", "")
        status = tc.get("status", "")
        runtime = tc.get("runtime_seconds", "")

        # Truncate long outputs for readability
        output_preview = str(output)[:500]
        if len(str(output)) > 500:
            output_preview += "..."

        # Format args nicely
        try:
            if isinstance(args, str):
                args_dict = json.loads(args)
            else:
                args_dict = args
            args_formatted = json.dumps(args_dict, indent=2)
        except (json.JSONDecodeError, TypeError):
            args_formatted = str(args)

        # Build collapsible block with <details>/<summary> tags
        lines.append(f"{i}. `{tool_name}`")
        lines.append(f"<details>")
        lines.append(f"<summary>Show details</summary>\n")
        lines.append(f"**Agent:** `{agent}`\n")
        if status:
            lines.append(f"**Status:** `{status}`\n")
        if runtime:
            lines.append(f"**Runtime:** `{runtime}s`\n")
        lines.append(f"**Args:**")
        lines.append(f"```json\n{args_formatted}\n```\n")
        lines.append(f"**Output:**")
        lines.append(f"```\n{output_preview}\n```")
        lines.append(f"</details>\n")

    return "\n".join(lines)


def transform_messages(messages, agent_models):
    """Transform conversation messages to include model info in role names.
    
    Handles real trace format where:
    - User messages have role "user"
    - Assistant messages have role "assistant" with content like "[agent_name] ..."
    
    Args:
        messages: List of message dicts with 'role' and 'content'
        agent_models: Dict mapping agent name -> model name
    
    Returns:
        List of messages with model names appended to agent roles
    """
    result = []
    for msg in messages:
        role = msg.get("role", "unknown")
        content = msg.get("content", "")
        
        if role == "assistant":
            # Extract agent name from content pattern: [agent_name] or `[agent_name]`
            agent_match = re.search(r'\[(\w+)\]', content)
            if agent_match:
                agent_name = agent_match.group(1)
                model = agent_models.get(agent_name, "unknown")
                role = f"{agent_name} ({model})"
            elif agent_models:
                # Fallback: use first agent if pattern not found
                agent_name = list(agent_models.keys())[0]
                model = agent_models[agent_name]
                role = f"{agent_name} ({model})"
        
        result.append({"role": role, "content": content})
    return result


def download_and_prepare(pair_info):
    """Download source items and prepare comparison data structure.
    
    Args:
        pair_info: Tuple of (item_a, item_b, query_id)
    
    Returns:
        Dict with 'query_id', 'data' (comparison content), and 'metadata'
    """
    item_a, item_b, query_id = pair_info
    
    # Download both trace files
    content_a = json.loads(item_a.download(save_locally=False).getvalue())
    content_b = json.loads(item_b.download(save_locally=False).getvalue())
    
    # Extract agent configurations from models_by_agent
    agent_models_a = extract_agent_models(content_a)
    agent_models_b = extract_agent_models(content_b)
    
    # Get conversation messages (real traces use "conversation" not "messages")
    messages_a = content_a.get("conversation", content_a.get("messages", []))
    messages_b = content_b.get("conversation", content_b.get("messages", []))
    
    # Get tool calls
    tool_calls_a = content_a.get("tool_calls", [])
    tool_calls_b = content_b.get("tool_calls", [])
    
    # Transform tool calls to structured format
    transformed_tool_calls_a = transform_tool_calls(tool_calls_a)
    transformed_tool_calls_b = transform_tool_calls(tool_calls_b)
    
    # Build comparison data matching sample_comparison_data.json format
    return {
        'query_id': query_id,
        'data': {
            "instructions": (
                "Compare the two conversation traces below. Each trace shows a multi-agent "
                "conversation where different AI models collaborate to answer the user's query.\n\n"
                "**Your task:** Determine which trace provides a better overall response "
                "considering accuracy, helpfulness, and quality of the agent collaboration."
            ),
            "trace_a_models": make_model_table(agent_models_a, "Trace A"),
            "conversation_a": transform_messages(messages_a, agent_models_a),
            "tool_calls_a": transformed_tool_calls_a,
            "trace_a_tool_calls": format_tool_calls_markdown(transformed_tool_calls_a, "Trace A"),
            "trace_b_models": make_model_table(agent_models_b, "Trace B"),
            "conversation_b": transform_messages(messages_b, agent_models_b),
            "tool_calls_b": transformed_tool_calls_b,
            "trace_b_tool_calls": format_tool_calls_markdown(transformed_tool_calls_b, "Trace B"),
            "source_query_id": query_id,
            "trace_a_source": item_a.filename,
            "trace_b_source": item_b.filename
        },
        'metadata': {
            'user': {
                'source_query_id': query_id,
                'trace_a_source': item_a.filename,
                'trace_b_source': item_b.filename,
                'trace_a_agents': agent_models_a,
                'trace_b_agents': agent_models_b,
                'item_a_id': item_a.id,
                'item_b_id': item_b.id
            }
        }
    }


def upload_item(item_info):
    """Upload a prepared comparison item to the output dataset.
    
    Args:
        item_info: Dict from download_and_prepare()
    
    Returns:
        Uploaded Dataloop item
    """
    buffer = io.BytesIO(json.dumps(item_info['data'], indent=2).encode('utf-8'))
    buffer.name = f"comparison_{item_info['query_id']}.json"
    buffer.seek(0)
    return output_dataset.items.upload(
        local_path=buffer,
        remote_path=OUTPUT_DIR,
        item_metadata=item_info['metadata']
    )

print("Helper functions defined ✓")

### 4b: Download & Transform Traces

Downloads both traces for each pair in parallel, extracts model configurations, and transforms the conversation data into the comparison format.

In [7]:
# Download and prepare all pairs in parallel
print(f"Downloading and preparing {len(pairs)} pairs...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    prepared_items = list(tqdm(
        executor.map(download_and_prepare, pairs),
        total=len(pairs),
        desc="Preparing"
    ))

print(f"\nPrepared {len(prepared_items)} comparison items")

Preparing: 100%|██████████| 106/106 [00:08<00:00, 12.62it/s]


Prepared 106 comparison items


### 4c: Upload to Evaluation Studio

Uploads all prepared comparison items to Dataloop in parallel. Each item is configured with Evaluation Studio metadata for the comparison layout.

In [8]:
# Upload all prepared items in parallel
print(f"Uploading to {OUTPUT_DATASET_NAME}{OUTPUT_DIR}...")

created_items = []
failed_items = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(upload_item, item): item['query_id'] for item in prepared_items}
    
    with tqdm(total=len(futures), desc="Uploading") as pbar:
        for future in as_completed(futures):
            query_id = futures[future]
            try:
                created_items.append(future.result())
            except Exception as e:
                failed_items.append((query_id, str(e)))
            pbar.update(1)

# Summary
print(f"\n{'='*50}")
print(f"Created: {len(created_items)} items")
if failed_items:
    print(f"Failed: {len(failed_items)} items")
    for qid, err in failed_items[:5]:
        print(f"  - {qid}: {err}")
print(f"Location: {OUTPUT_DATASET_NAME}{OUTPUT_DIR}")
print(f"{'='*50}")

Uploading to model-comparisons/20260104.1501_csv_20251223_101531_vs_csv_20251226_111400...


Uploading: 100%|██████████| 106/106 [00:09<00:00, 11.16it/s]


Created: 106 items
Location: model-comparisons/20260104.1501_csv_20251223_101531_vs_csv_20251226_111400


---
## Step 5: Create Annotation Task

Creates a Dataloop task and assigns it to annotators. By default, assigns to the currently logged-in user.

**Optional:** Edit `ASSIGNEE_IDS` below to add more annotators.

In [9]:
# Create annotation task for the comparison items
TASK_NAME = f"Model Comparison {date_str}"

# Default to current logged-in user
user_info = dl.info()
ASSIGNEE_IDS = [user_info['user_email']]  # Add/modify annotator emails as needed

# Create filter for items in the output directory
filters = dl.Filters()
filters.add(field='dir', values=OUTPUT_DIR)

# Create the task with filters
task = output_dataset.tasks.create(
    task_name=TASK_NAME,
    assignee_ids=ASSIGNEE_IDS,
    filters=filters
)

# Generate link to the task on the platform

print("=" * 60)
print("ANNOTATION TASK CREATED")
print("=" * 60)
print(f"\nTask Name: {TASK_NAME}")
print(f"Task ID: {task.id}")
print(f"Assignees: {', '.join(ASSIGNEE_IDS)}")
print(f"Items: {len(created_items)} comparison items\n")
print("Next steps:")
print("  1. Open the task link below")
print("  2. For each item, compare Trace A and Trace B")
print("  3. Select which trace provides the better response")
print("  4. Submit your annotation\n")
print(f"Task URL:\n  {task.open_in_web()}\n")
print("=" * 60)

input("\nPress Enter after completing all annotations on the platform to continue...")

ANNOTATION TASK CREATED

Task Name: Model Comparison 20260104.1501
Task ID: 695a64628f9d32791a242b57
Assignees: yaya.t@dataloop.ai
Items: 106 comparison items

Next steps:
  1. Open the task link below
  2. For each item, compare Trace A and Trace B
  3. Select which trace provides the better response
  4. Submit your annotation

Task URL:
  None



''

---
---
# PART 2: Analyze Results
---

**Run the cells below after annotators have completed the task.**

The cells will:
1. Load all comparison items from the output directory
2. Collect annotations (preferences) from each item
3. Summarize results showing which trace configuration won

---
## Step 6: Load Comparison Items

Fetches all comparison items created in Part 1.

In [10]:
# Load comparison items from output directory
filters = dl.Filters()
filters.add(field='dir', values=OUTPUT_DIR)
filters.add(field='name', values='*comparison_*')

comparison_items = list(output_dataset.items.list(filters=filters).all())
print(f"Found {len(comparison_items)} comparison items in {OUTPUT_DATASET_NAME}{OUTPUT_DIR}")

Found 106 comparison items in model-comparisons/20260104.1501_csv_20251223_101531_vs_csv_20251226_111400


---
## Step 7: Collect Annotations

Downloads each comparison item and extracts the annotator's preference (Trace A vs Trace B).

In [11]:
annotations_data = []

for item in comparison_items:
    item_data = json.loads(item.download(save_locally=False).getvalue())
    annotations = item.annotations.list()
    
    # Get agent info from item metadata (not from data payload)
    user_metadata = item.metadata.get('user', {})
    
    for annotation in annotations:
        form_data = annotation.attributes if hasattr(annotation, 'attributes') else {}
        
        annotations_data.append({
            'item_name': item.name,
            'query_id': item_data.get('source_query_id'),
            'trace_a_agents': user_metadata.get('trace_a_agents', {}),
            'trace_b_agents': user_metadata.get('trace_b_agents', {}),
            'preferred_model': form_data.get('preferred_model'),
            'annotator': annotation.creator
        })

print(f"Collected {len(annotations_data)} annotations")

Collected 1 annotations


---
## Step 8: Summarize Results

Aggregates annotations and displays:
- **Overall winner**: Which trace was preferred more often
- **Model performance by role**: Win rate for each model in each agent role

In [12]:
if not annotations_data:
    print("No annotations found yet. Complete the annotation task first.")
else:
    # Count preferences
    counts = defaultdict(int)
    model_wins = defaultdict(lambda: defaultdict(int))
    model_appearances = defaultdict(lambda: defaultdict(int))
    
    for ann in annotations_data:
        pref = ann.get('preferred_model')
        if not pref:
            continue
            
        counts[pref] += 1
        
        # Track model performance
        for role, model in ann.get('trace_a_agents', {}).items():
            model_appearances[role][model] += 1
            if pref == 'trace_a':
                model_wins[role][model] += 1
                
        for role, model in ann.get('trace_b_agents', {}).items():
            model_appearances[role][model] += 1
            if pref == 'trace_b':
                model_wins[role][model] += 1
    
    total = sum(counts.values())
    
    # Print summary
    print("=" * 50)
    print("ANNOTATION SUMMARY")
    print("=" * 50)
    print(f"\nTotal annotations: {total}\n")
    
    print("--- Overall Preferences ---")
    for pref, count in sorted(counts.items(), key=lambda x: -x[1]):
        pct = count / total * 100
        bar = "#" * int(pct / 5)
        print(f"  {pref:15} : {count:4} ({pct:5.1f}%) {bar}")
    
    winner = max(counts, key=counts.get)
    print(f"\n*** WINNER: {winner} ***")
    
    print("\n--- Model Performance by Role ---")
    for role in model_appearances:
        print(f"\n  {role}:")
        for model in model_appearances[role]:
            wins = model_wins[role].get(model, 0)
            apps = model_appearances[role][model]
            rate = wins / apps * 100 if apps > 0 else 0
            print(f"    {model}: {wins}/{apps} wins ({rate:.1f}%)")
    
    print("\n" + "=" * 50)

ANNOTATION SUMMARY

Total annotations: 0

--- Overall Preferences ---


ValueError: max() iterable argument is empty

---
## Step 9: Raw Data (Optional)

View the raw annotation data for further analysis or export.

In [ ]:
# Display raw annotation data as a table
if annotations_data:
    df = pd.DataFrame(annotations_data)
    display(df)
else:
    print("No annotations to display.")